# Power Analysis: Why Most A/B Tests Are Lying to You

Based on the paper:  
**"Power Analysis is Essential: High-Powered Tests Suggest Minimal to No Effect of Rounded Shapes on Click-Through Rates"**  
Kohavi et al. (arXiv: 2512.24521)

---
### What we'll cover:
1. What is Statistical Power?
2. The Winner's Curse — why published results are inflated
3. Replicating the paper's findings visually
4. How to do Power Analysis before your experiment
5. Minimum sample size calculator


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm, ttest_ind

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 13
np.random.seed(42)

---
## 1. What is Statistical Power?

**Statistical power** is the probability that your test *correctly detects a real effect* when one exists.

- Power = 1 - β (where β = probability of a false negative)
- Standard target: **power ≥ 0.80** (80%)
- Power depends on: **sample size**, **effect size**, and **significance level (α)**

> Think of it like a metal detector. A weak detector (small sample) misses small coins. A strong detector (large sample) finds even tiny ones.

In [ ]:
# --- Visualize: Power vs Sample Size for different effect sizes ---

def compute_power(n, effect_size, alpha=0.05):
    """Compute power of a two-sample t-test."""
    se = np.sqrt(2 / n)
    critical_value = norm.ppf(1 - alpha / 2)
    ncp = effect_size / se  # non-centrality parameter
    power = 1 - norm.cdf(critical_value - ncp) + norm.cdf(-critical_value - ncp)
    return power

sample_sizes = np.arange(10, 5000, 10)
effect_sizes = {'Large (d=0.8)': 0.8, 'Medium (d=0.5)': 0.5,
                'Small (d=0.2)': 0.2, 'Tiny (d=0.05)': 0.05}
colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

fig, ax = plt.subplots()
for (label, es), color in zip(effect_sizes.items(), colors):
    powers = [compute_power(n, es) for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=label, color=color, linewidth=2.5)

ax.axhline(0.80, color='black', linestyle='--', linewidth=1.5, label='80% Power threshold')
ax.set_xlabel('Sample Size (per group)')
ax.set_ylabel('Statistical Power')
ax.set_title('Statistical Power vs. Sample Size\nby Effect Size (Cohen\'s d)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
ax.fill_between(sample_sizes, 0, 0.80, alpha=0.05, color='red')
ax.text(4200, 0.35, 'Danger\nZone', color='red', alpha=0.6, ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('power_vs_sample_size.png', dpi=150)
plt.show()
print("Key insight: A tiny effect (d=0.05) needs tens of thousands of samples to detect reliably.")

---
## 2. The Winner's Curse

When a study is **underpowered**, most experiments fail to reach significance. The few that do are **statistical flukes** — extreme overestimates of the true effect.

This is the **Winner's Curse**: published significant results from underpowered studies are inflated by design.

> The original rounded corners study had ~30 participants. It found +55% CTR.  
> The replication had 60,000+ users. It found ~0.5% — not significant.

In [ ]:
# --- Simulate the Winner's Curse ---

def simulate_winners_curse(true_effect, n_per_group, n_simulations=10000, alpha=0.05):
    """Run many underpowered experiments, collect only 'significant' results."""
    all_effects = []
    significant_effects = []

    for _ in range(n_simulations):
        control = np.random.normal(0, 1, n_per_group)
        treatment = np.random.normal(true_effect, 1, n_per_group)
        observed_effect = treatment.mean() - control.mean()
        _, p = ttest_ind(control, treatment)
        all_effects.append(observed_effect)
        if p < alpha:
            significant_effects.append(observed_effect)

    return np.array(all_effects), np.array(significant_effects)

true_effect = 0.05  # tiny true effect (like rounded corners)
all_effects, sig_effects = simulate_winners_curse(true_effect, n_per_group=30)

fig, ax = plt.subplots()
ax.hist(all_effects, bins=80, color='#bdc3c7', alpha=0.7, label='All experiments', density=True)
ax.hist(sig_effects, bins=40, color='#e74c3c', alpha=0.8, label=f'Significant results only (n={len(sig_effects)})', density=True)
ax.axvline(true_effect, color='#2ecc71', linewidth=2.5, linestyle='--', label=f'True effect = {true_effect}')
ax.axvline(np.mean(sig_effects), color='#e74c3c', linewidth=2.5, linestyle='-',
           label=f'Mean published effect = {np.mean(sig_effects):.2f}')
ax.set_xlabel('Observed Effect Size')
ax.set_ylabel('Density')
ax.set_title("The Winner's Curse\nUnderpowered studies inflate effect sizes", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('winners_curse.png', dpi=150)
plt.show()

inflation = np.mean(sig_effects) / true_effect
print(f"True effect:              {true_effect}")
print(f"Mean 'published' effect:  {np.mean(sig_effects):.2f}")
print(f"Inflation factor:         {inflation:.1f}x")
print(f"Only {len(sig_effects)/100:.1f}% of experiments were significant (most were underpowered)")

---
## 3. Original Claim vs. Replications

The paper compared the original study's claimed effect against 5 high-powered replications.

| Study | Sample Size | Effect (CTR lift) | Significant? |
|---|---|---|---|
| Original (2023) | ~30 | +55% | Yes (p=0.037) |
| Replication 1 | 60,000+ | ~0.5% | No |
| Replication 2 | 60,000+ | ~0.3% | No |
| Replication 3 | 60,000+ | ~0.6% | No |
| Evidoo Rep. 1 | 60,000+ | ~0.4% | No |
| Evidoo Rep. 2 | 60,000+ | ~0.2% | No |

In [ ]:
# --- Original vs Replication Bar Chart ---

studies = ['Original\n(2023)', 'Replication\n1', 'Replication\n2', 'Replication\n3',
           'Evidoo\nRep. 1', 'Evidoo\nRep. 2']
effects = [55.0, 0.5, 0.3, 0.6, 0.4, 0.2]
errors  = [28.0, 0.4, 0.4, 0.5, 0.3, 0.3]  # approximate 95% CI half-widths
colors  = ['#e74c3c'] + ['#3498db'] * 5
significant = [True, False, False, False, False, False]

fig, ax = plt.subplots()
bars = ax.bar(studies, effects, color=colors, yerr=errors, capsize=6,
              error_kw={'linewidth': 2}, edgecolor='white', linewidth=1.5)

for bar, sig in zip(bars, significant):
    label = '* Significant' if sig else 'Not Significant'
    color = '#c0392b' if sig else '#7f8c8d'
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (2 if sig else 0.15),
            label, ha='center', va='bottom', fontsize=10, color=color, fontweight='bold')

ax.set_ylabel('Click-Through Rate Lift (%)')
ax.set_title('Rounded Corners Effect: Original Claim vs. High-Powered Replications',
             fontweight='bold')
orig_patch = mpatches.Patch(color='#e74c3c', label='Original study (~30 users)')
rep_patch  = mpatches.Patch(color='#3498db', label='Replications (60,000+ users each)')
ax.legend(handles=[orig_patch, rep_patch])
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('original_vs_replications.png', dpi=150)
plt.show()
print("The original 55% claim was ~100x larger than the true effect.")

---
## 4. How to Do Power Analysis BEFORE Your Experiment

**Steps:**
1. Decide your **minimum detectable effect (MDE)** — the smallest effect that would matter to you
2. Set your **significance level** (α = 0.05)
3. Set your **desired power** (1-β = 0.80)
4. Calculate the **required sample size**

Formula for two-sample test:  
$$n = \frac{2(z_{\alpha/2} + z_{\beta})^2}{d^2}$$

where $d$ = Cohen's d (effect size / pooled std)

In [ ]:
# --- Sample Size Calculator ---

def required_sample_size(effect_size, alpha=0.05, power=0.80):
    """Calculate required sample size per group for a two-sample t-test."""
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    n = 2 * ((z_alpha + z_beta) / effect_size) ** 2
    return int(np.ceil(n))

print("Required sample size per group (alpha=0.05, power=80%)")
print("-" * 50)
examples = [
    ("Large effect  (d=0.80) — big UI change",      0.80),
    ("Medium effect (d=0.50) — moderate change",    0.50),
    ("Small effect  (d=0.20) — minor tweak",        0.20),
    ("Tiny effect   (d=0.05) — rounded corners!",   0.05),
]
for label, d in examples:
    n = required_sample_size(d)
    print(f"{label}: n = {n:,} per group ({2*n:,} total)")

In [ ]:
# --- Interactive: Power Analysis Heatmap ---
# How does required sample size change across effect sizes and power levels?

effect_sizes_range = np.linspace(0.05, 0.8, 20)
power_levels       = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

grid = np.array([[required_sample_size(es, power=pw)
                  for es in effect_sizes_range]
                 for pw in power_levels])

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(grid, aspect='auto', cmap='YlOrRd_r',
               norm=plt.matplotlib.colors.LogNorm(vmin=grid.min(), vmax=grid.max()))

ax.set_xticks(range(len(effect_sizes_range)))
ax.set_xticklabels([f'{e:.2f}' for e in effect_sizes_range], rotation=45, ha='right')
ax.set_yticks(range(len(power_levels)))
ax.set_yticklabels([f'{p*100:.0f}%' for p in power_levels])
ax.set_xlabel("Effect Size (Cohen's d)")
ax.set_ylabel('Desired Power')
ax.set_title('Required Sample Size per Group\n(log scale — green = fewer samples needed)',
             fontweight='bold')

for i in range(len(power_levels)):
    for j in range(len(effect_sizes_range)):
        val = grid[i, j]
        text = f'{val:,}' if val < 10000 else f'{val//1000}k'
        ax.text(j, i, text, ha='center', va='center', fontsize=7,
                color='white' if val > 2000 else 'black')

plt.colorbar(im, ax=ax, label='Sample size per group')
plt.tight_layout()
plt.savefig('sample_size_heatmap.png', dpi=150)
plt.show()

---
## 5. Key Takeaways

| Lesson | What to Do |
|---|---|
| Small samples = inflated effects | Always power your study before running it |
| Winner's curse is real | Be skeptical of dramatic results from small studies |
| p < 0.05 ≠ large effect | Report effect sizes and confidence intervals |
| Replication is the gold standard | Try to replicate findings before acting on them |
| Even 55% claims can be ~0% | Run your own high-powered A/B test |

### Quick Power Analysis Checklist
- [ ] Define your minimum detectable effect (MDE)
- [ ] Set α = 0.05 and target power = 0.80 (or higher)
- [ ] Calculate required sample size **before** running the experiment
- [ ] Do NOT stop the experiment early (optional stopping inflates false positives)
- [ ] Report confidence intervals alongside p-values

In [ ]:
# --- Bonus: A/B Test Simulation with correct sample size ---
# Demonstrate what happens when you run a properly powered experiment

true_effect = 0.05
n_required  = required_sample_size(true_effect, alpha=0.05, power=0.80)
print(f"For effect size d=0.05, required n per group: {n_required:,}")
print(f"Total participants needed: {2*n_required:,}")
print()

# Underpowered experiment (original study)
n_small = 15  # ~30 total
control_small   = np.random.normal(0, 1, n_small)
treatment_small = np.random.normal(true_effect, 1, n_small)
_, p_small = ttest_ind(control_small, treatment_small)
obs_effect_small = treatment_small.mean() - control_small.mean()

# Well-powered experiment (replication)
control_large   = np.random.normal(0, 1, n_required)
treatment_large = np.random.normal(true_effect, 1, n_required)
_, p_large = ttest_ind(control_large, treatment_large)
obs_effect_large = treatment_large.mean() - control_large.mean()

print(f"{'Experiment':<30} {'n/group':>10} {'Observed Effect':>18} {'p-value':>12} {'Conclusion':>20}")
print("-" * 95)
print(f"{'Underpowered (original)':<30} {n_small:>10,} {obs_effect_small:>18.3f} {p_small:>12.3f} {'SIGNIFICANT' if p_small<0.05 else 'not significant':>20}")
print(f"{'Well-powered (replication)':<30} {n_required:>10,} {obs_effect_large:>18.3f} {p_large:>12.3f} {'SIGNIFICANT' if p_large<0.05 else 'not significant':>20}")
print(f"\nTrue effect: {true_effect}")